# Operator Comparison: LLM vs OWL Belief Graph

This notebook demonstrates **ontology-grounded operator comparison** between two models:

- **Model A** (neural): an LLM queried via Ollama
- **Model B** (symbolic): an OWL belief graph that answers by querying SPARQL/pyshacl

We measure divergence via $L(x, A(x), B(x))$ — scoring both operators against ground-truth ontological facts at each probe point $x$. In **imitation mode**, B updates its belief graph after observing A's responses, performing online system identification: learning a symbolic model that imitates a neural operator.

**Sparse mode**: B starts with only the TBox (class hierarchy, disjointness) and direct instance types — no property assertions or entailed types. This forces B to *learn* relational structure from A rather than just confirming what it already knows.

## Configuration

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from IPython.display import display, HTML

from apollo_eval.ontology_loader import OntologyLoader
from apollo_eval.vocab import OntologyVocabularyBuilder
from apollo_eval.question import Question
from apollo_eval.question_loader import build_question_bank
from apollo_eval.model_client import OllamaClient
from apollo_eval.ontology_client import OntologyClient
from apollo_eval.comparison_engine import ComparisonEngine
from apollo_eval.divergence import ProbeResult

# --- Experiment parameters ---
N_EPOCHS = 5
SEED = 42
SIMILARITY_THRESHOLD = 1.0
MODEL = "llama3.2:1b"
SPARSE = True          # B starts with minimal knowledge
IMITATION = True       # B learns from A
OUTPUT_DIR = "trajectories/comparison_notebook"
SCENARIO_IRI = "http://example.org/scenario/measles_epidemic/"

# Set to a path to load a previous run instead of running a new one
LOAD_EXPERIMENT = None  # e.g. "trajectories/comparison_notebook/compare_xxx.json"

# --- Plot style ---
sns.set_theme(style="whitegrid", font_scale=1.1)
TYPE_COLORS = {"CM": "#4C72B0", "PR": "#DD8452", "DJ": "#55A868"}
TYPE_LABELS = {"CM": "Class Membership", "PR": "Property Restriction", "DJ": "Disjointness"}

## Load Ontology and Build Questions

In [ ]:
loader = OntologyLoader()
graph = loader.load_graph()
graph = loader.apply_reasoning(graph)
ontology = OntologyVocabularyBuilder(graph).build(
    similarity_threshold=SIMILARITY_THRESHOLD,
)
Question.ontology = ontology
questions = build_question_bank(ontology)
print(f"{len(questions)} questions ready.")

## Create Operators and Inspect B's Initial State

Before running, let's see what B knows and doesn't know in sparse mode.

In [ ]:
from apollo_eval.evaluation import question_type as get_question_type

client_b_preview = OntologyClient(ontology, questions, sparse=SPARSE)

b_mode = "sparse (TBox + direct types)" if SPARSE else "full (ground truth copy)"
print(f"Model B mode: {b_mode}")
print(f"Ground truth graph: {len(ontology.graph)} triples")
print(f"B's belief graph:   {len(client_b_preview.belief_graph)} triples")
print(f"Stripped:            {len(ontology.graph) - len(client_b_preview.belief_graph)} triples\n")

# Show B's initial answers for all 25 questions
rows = []
for q in questions:
    b_resp = client_b_preview.complete([], q.prompt)
    b_score = q.score(b_resp)
    rows.append({
        "question_id": q.question_id[:8] + "...",
        "type": get_question_type(q),
        "prompt": q.prompt[:70] + ("..." if len(q.prompt) > 70 else ""),
        "B_response": b_resp,
        "B_score": f"{b_score.score:.0f}" if b_score.parse_success else "N/A",
    })

initial_df = pd.DataFrame(rows)
print("B's initial answers (before observing A):")
print(f"  Correct: {sum(1 for r in rows if r['B_score'] == '1')}/{len(rows)}")
print(f"  Wrong:   {sum(1 for r in rows if r['B_score'] == '0')}/{len(rows)}")
print(f"  N/A:     {sum(1 for r in rows if r['B_score'] == 'N/A')}/{len(rows)}")
display(initial_df.style.hide(axis="index"))

## Run Comparison Experiment

Probes both A and B with all 25 questions for N_EPOCHS epochs. A's history resets each epoch; B's belief graph persists and accumulates learning.

In [ ]:
if LOAD_EXPERIMENT:
    print(f"Loading from {LOAD_EXPERIMENT}...")
    with open(LOAD_EXPERIMENT) as f:
        saved = json.load(f)

    # Reconstruct trajectory from saved JSON
    from apollo_eval.ontology import ScoreResult
    from apollo_eval.divergence import DivergenceTrajectory, ProbeResult

    trajectory = DivergenceTrajectory(metadata=saved)
    question_map = {q.question_id: q for q in questions}
    for p in saved["probes"]:
        q = question_map[p["question_id"]]
        a_score = q.score(p["a_response"])
        b_score = q.score(p["b_response"])
        trajectory.probes.append(ProbeResult(
            question_id=p["question_id"],
            question_type=p["question_type"],
            epoch=p["epoch"],
            probe_index=p["probe_index"],
            a_response=p["a_response"],
            b_response=p["b_response"],
            a_score=a_score,
            b_score=b_score,
            divergent=p["divergent"],
            belief_update=p.get("belief_update"),
        ))
    print(f"Loaded {len(trajectory.probes)} probes.")
else:
    client_a = OllamaClient(model=MODEL)
    client_b = OntologyClient(ontology, questions, sparse=SPARSE)

    print(f"Operator A: {client_a.model} (LLM)")
    print(f"Operator B: OWL belief graph ({'sparse' if SPARSE else 'full'})")
    print(f"Epochs: {N_EPOCHS}  |  Imitation: {IMITATION}\n")

    probe_log = []
    def log_probe(probe):
        a_str = f"{probe.a_score.score:.0f}" if probe.a_score.parse_success else "?"
        b_str = f"{probe.b_score.score:.0f}" if probe.b_score.parse_success else "?"
        div = "DIV" if probe.divergent else "   "
        upd = " +update" if probe.belief_update else ""
        print(f"  [{probe.epoch}:{probe.probe_index:02d}] A={a_str} B={b_str} {div}{upd}")
        probe_log.append(probe)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    engine = ComparisonEngine(
        questions=questions,
        client_a=client_a,
        client_b=client_b,
        n_epochs=N_EPOCHS,
        seed=SEED,
        scenario_iri=SCENARIO_IRI,
        imitation=IMITATION,
        on_probe=log_probe,
        metadata={
            "model_a": client_a.model,
            "model_b": "owl_belief_graph",
            "model_b_mode": "sparse" if SPARSE else "full",
        },
    )

    print(f"Running {N_EPOCHS} epochs...")
    trajectory = engine.run()
    path = engine.save(OUTPUT_DIR)
    print(f"\nDone. Saved to {path}")

## Experiment Summary

In [ ]:
s = trajectory.summary()
df = trajectory.to_dataframe()

fmt = lambda v: f"{v:.1%}" if v == v else "N/A"

summary_rows = [
    ("Total probes", str(s["total_probes"])),
    ("Epochs", str(s["n_epochs"])),
    ("Questions per epoch", str(s["n_questions"])),
    ("", ""),
    ("A parse rate", fmt(s["a_parse_rate"])),
    ("B parse rate", fmt(s["b_parse_rate"])),
    ("A consistency (vs ground truth)", fmt(s["a_consistency_rate"])),
    ("B consistency (vs ground truth)", fmt(s["b_consistency_rate"])),
    ("", ""),
    ("Overall divergence rate", fmt(s["overall_divergence_rate"])),
    ("Total belief updates", str(s["total_belief_updates"])),
]

summary_df = pd.DataFrame(summary_rows, columns=["Metric", "Value"])
summary_df = summary_df[summary_df["Metric"] != ""]
display(summary_df.style.hide(axis="index"))

## Convergence Curve: Divergence by Epoch

The key plot: does B's divergence from A decrease over epochs as B learns from A's responses? If imitation is working, we expect the divergence rate to fall and B's consistency to rise toward A's level.

In [ ]:
by_epoch = trajectory.divergence_by_epoch()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Divergence rate
ax = axes[0]
ax.plot(by_epoch.index, by_epoch["divergence_rate"], marker="o", linewidth=2, color="#D94A3D")
ax.fill_between(by_epoch.index, by_epoch["divergence_rate"], alpha=0.15, color="#D94A3D")
ax.set_xlabel("Epoch")
ax.set_ylabel("Divergence Rate")
ax.set_ylim(-0.05, 1.05)
ax.set_title("A-B Divergence Rate")
ax.set_xticks(by_epoch.index)

# 2. Consistency rates (A vs B vs ground truth)
ax = axes[1]
ax.plot(by_epoch.index, by_epoch["a_consistency"], marker="s", linewidth=2, color="#4C72B0", label="A (LLM)")
ax.plot(by_epoch.index, by_epoch["b_consistency"], marker="^", linewidth=2, color="#55A868", label="B (OWL)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Consistency Rate (vs ground truth)")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Consistency vs Ground Truth")
ax.set_xticks(by_epoch.index)
ax.legend()

# 3. Belief updates per epoch
ax = axes[2]
ax.bar(by_epoch.index, by_epoch["n_updates"], color="#8B6BB0")
ax.set_xlabel("Epoch")
ax.set_ylabel("Belief Updates")
ax.set_title("B's Belief Updates per Epoch")
ax.set_xticks(by_epoch.index)

plt.suptitle("Convergence Dynamics", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nPer-epoch statistics:")
display(by_epoch)

## Divergence by Question Type

How does divergence distribute across the three question types? In sparse mode, we expect PR (property restriction) to dominate initial divergence since B starts with no relational knowledge.

In [ ]:
by_type = trajectory.divergence_by_type()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

qt_order = [t for t in ["CM", "PR", "DJ"] if t in by_type.index]

# Divergence rate by type
ax = axes[0]
ax.bar(qt_order, [by_type.loc[t, "divergence_rate"] for t in qt_order],
       color=[TYPE_COLORS[t] for t in qt_order])
ax.set_ylabel("Divergence Rate")
ax.set_ylim(0, 1.05)
ax.set_title("Divergence Rate by Type")

# A vs B consistency by type
ax = axes[1]
x = np.arange(len(qt_order))
w = 0.35
ax.bar(x - w/2, [by_type.loc[t, "a_consistency"] for t in qt_order], w,
       label="A (LLM)", color="#4C72B0")
ax.bar(x + w/2, [by_type.loc[t, "b_consistency"] for t in qt_order], w,
       label="B (OWL)", color="#55A868")
ax.set_xticks(x)
ax.set_xticklabels(qt_order)
ax.set_ylabel("Consistency (vs ground truth)")
ax.set_ylim(0, 1.05)
ax.set_title("Consistency by Type")
ax.legend()

plt.tight_layout()
plt.show()

print("\nPer-type statistics:")
display(by_type)

## Convergence by Question Type Over Epochs

Track how divergence within each question type evolves across epochs. This decomposition shows whether B learns some categories of knowledge faster than others.

In [ ]:
# Divergence by type over epochs
type_epoch = df.groupby(["epoch", "question_type"]).agg(
    divergence_rate=("divergent", "mean"),
    b_consistency=("b_score", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for qt in ["CM", "PR", "DJ"]:
    subset = type_epoch[type_epoch["question_type"] == qt]
    if subset.empty:
        continue
    axes[0].plot(subset["epoch"], subset["divergence_rate"], marker="o",
                 linewidth=2, color=TYPE_COLORS[qt], label=TYPE_LABELS[qt])
    axes[1].plot(subset["epoch"], subset["b_consistency"], marker="o",
                 linewidth=2, color=TYPE_COLORS[qt], label=TYPE_LABELS[qt])

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Divergence Rate")
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title("Divergence Rate by Type Over Epochs")
axes[0].legend()

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("B Consistency (vs ground truth)")
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_title("B's Learning Curve by Type")
axes[1].legend()

plt.tight_layout()
plt.show()

## Probe Heatmap: A-B Agreement Over Epochs

Each row is a question, each column is an epoch. Color encodes the outcome:
- **Green**: A and B agree, both correct
- **Yellow**: A and B agree, both wrong
- **Red**: A and B diverge
- **Gray**: parse failure (can't determine)

In [ ]:
# Build grid: rows=questions, cols=epochs
# Values: 2=agree+correct, 1=agree+wrong, 0=diverge, -1=parse fail
q_ids = [q.question_id for q in questions]
n_epochs = df["epoch"].nunique()

grid = np.full((len(q_ids), n_epochs), np.nan)
for _, row in df.iterrows():
    qi = q_ids.index(row["question_id"]) if row["question_id"] in q_ids else None
    if qi is None:
        continue
    epoch = int(row["epoch"])
    if not row["a_parse_success"] or not row["b_parse_success"]:
        grid[qi, epoch] = -1  # parse fail
    elif row["divergent"]:
        grid[qi, epoch] = 0  # diverge
    elif row["a_score"] == 1.0 and row["b_score"] == 1.0:
        grid[qi, epoch] = 2  # agree + correct
    else:
        grid[qi, epoch] = 1  # agree + wrong

cmap = ListedColormap(["#999999", "#D94A3D", "#F0C05A", "#4CAF50"])
bounds = [-1.5, -0.5, 0.5, 1.5, 2.5]
norm = plt.matplotlib.colors.BoundaryNorm(bounds, cmap.N)

# Question labels with type prefix
q_labels = [f"[{get_question_type(q)}] {q.prompt[:50]}..." for q in questions]

fig, ax = plt.subplots(figsize=(max(8, n_epochs * 1.5), max(6, len(q_ids) * 0.35)))
im = ax.imshow(grid, cmap=cmap, norm=norm, aspect="auto", interpolation="nearest")

ax.set_xlabel("Epoch")
ax.set_ylabel("Question")
ax.set_xticks(range(n_epochs))
ax.set_yticks(range(len(q_ids)))
ax.set_yticklabels(q_labels, fontsize=7)
ax.set_title("A-B Agreement Heatmap Over Epochs")

legend_elements = [
    Patch(facecolor="#4CAF50", label="Agree + Correct"),
    Patch(facecolor="#F0C05A", label="Agree + Wrong"),
    Patch(facecolor="#D94A3D", label="Divergent"),
    Patch(facecolor="#999999", label="Parse Failure"),
]
ax.legend(handles=legend_elements, loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
plt.tight_layout()
plt.show()

## Text-In / Text-Out: Example Interactions

Side-by-side comparison of A's and B's actual responses for selected probes, showing how B's answers change across epochs as it learns from A.

In [ ]:
def render_probe_card(probe, question_map):
    """Render a single probe as an HTML card."""
    q = question_map.get(probe.question_id)
    q_type = probe.question_type

    a_ok = probe.a_score.parse_success
    b_ok = probe.b_score.parse_success
    a_score_str = f"{probe.a_score.score:.0f}" if a_ok else "parse fail"
    b_score_str = f"{probe.b_score.score:.0f}" if b_ok else "parse fail"
    a_color = "#4CAF50" if a_ok and probe.a_score.score == 1.0 else "#D94A3D" if a_ok else "#999"
    b_color = "#4CAF50" if b_ok and probe.b_score.score == 1.0 else "#D94A3D" if b_ok else "#999"
    div_badge = '<span style="color:#D94A3D;font-weight:bold">DIVERGENT</span>' if probe.divergent else '<span style="color:#4CAF50">aligned</span>'
    upd_badge = f' <span style="color:#8B6BB0;font-size:0.85em">[belief {probe.belief_update["action"]}]</span>' if probe.belief_update else ""

    prompt_text = q.prompt if q else probe.question_id
    a_resp = probe.a_response.replace("<", "&lt;").replace(">", "&gt;")
    b_resp = probe.b_response.replace("<", "&lt;").replace(">", "&gt;")

    return f"""
    <div style="border:1px solid #333; border-radius:6px; padding:12px; margin:8px 0; background:#0d1018; font-family:monospace; font-size:0.85em">
      <div style="margin-bottom:8px">
        <span style="color:#888">epoch {probe.epoch} | probe {probe.probe_index:02d} | {q_type}</span>
        &nbsp; {div_badge}{upd_badge}
      </div>
      <div style="color:#ccc; margin-bottom:10px; font-family:Georgia,serif; font-size:1.05em">
        <strong>Q:</strong> {prompt_text}
      </div>
      <div style="display:flex; gap:12px">
        <div style="flex:1; background:#111520; border-radius:4px; padding:10px; border-left:3px solid #4C72B0">
          <div style="font-size:0.8em; color:#4C72B0; margin-bottom:4px">
            A (LLM) &mdash; score: <span style="color:{a_color}">{a_score_str}</span>
          </div>
          <div style="color:#bbb; white-space:pre-wrap; word-wrap:break-word">{a_resp[:500]}</div>
        </div>
        <div style="flex:1; background:#111520; border-radius:4px; padding:10px; border-left:3px solid #55A868">
          <div style="font-size:0.8em; color:#55A868; margin-bottom:4px">
            B (OWL) &mdash; score: <span style="color:{b_color}">{b_score_str}</span>
          </div>
          <div style="color:#bbb; white-space:pre-wrap; word-wrap:break-word">{b_resp}</div>
        </div>
      </div>
    </div>"""


question_map = {q.question_id: q for q in questions}

### Divergent Probes: Where A and B Disagree

These are the most interesting interactions -- cases where the LLM and the belief graph give different answers. In imitation mode, B updates its beliefs after each of these.

In [ ]:
divergent_probes = [p for p in trajectory.probes if p.divergent]
print(f"Total divergent probes: {len(divergent_probes)} / {len(trajectory.probes)}\n")

# Show up to 8 divergent probes, prioritizing first epoch
shown = sorted(divergent_probes, key=lambda p: (p.epoch, p.probe_index))[:8]
html = "<h4>Selected divergent interactions</h4>"
for p in shown:
    html += render_probe_card(p, question_map)

display(HTML(html))

### B's Learning Trajectory: Same Question Across Epochs

For a question where B initially got it wrong, track how B's response changes across epochs as it learns from A. This is the core demonstration of online imitation.

In [ ]:
# Find a question that was divergent in epoch 0 but converged later
# Group probes by question_id, look for the pattern: divergent early, aligned later
best_q = None
best_story = None

probes_by_q = {}
for p in trajectory.probes:
    probes_by_q.setdefault(p.question_id, []).append(p)

for qid, probes in probes_by_q.items():
    probes_sorted = sorted(probes, key=lambda p: p.epoch)
    if len(probes_sorted) < 2:
        continue
    # Was divergent in epoch 0?
    if probes_sorted[0].divergent:
        # Did it converge at some point?
        later_aligned = [p for p in probes_sorted[1:] if not p.divergent]
        if later_aligned:
            best_q = qid
            best_story = probes_sorted
            break

if best_story:
    q = question_map[best_q]
    html = f"<h4>Learning trajectory for: <em>{q.prompt[:80]}</em></h4>"
    html += f'<p style="color:#888; font-size:0.9em">Question type: {get_question_type(q)} | ID: {best_q[:16]}...</p>'
    for p in best_story:
        html += render_probe_card(p, question_map)
    display(HTML(html))
else:
    # Fallback: show any question across all epochs
    any_q = list(probes_by_q.keys())[0]
    story = sorted(probes_by_q[any_q], key=lambda p: p.epoch)
    q = question_map[any_q]
    html = f"<h4>Trajectory for: <em>{q.prompt[:80]}</em></h4>"
    for p in story:
        html += render_probe_card(p, question_map)
    display(HTML(html))

### Belief Updates Log

All graph modifications B made while imitating A. Each entry is a triple that was added to or retracted from B's belief graph.

In [ ]:
updates = [(p.epoch, p.question_id[:8], p.question_type, p.belief_update)
           for p in trajectory.probes if p.belief_update is not None]

if updates:
    update_rows = []
    for epoch, qid, qtype, upd in updates:
        triple = upd.get("triple", ("?", "?", "?"))
        # Shorten IRIs for readability
        short = lambda s: s.split("/")[-1] if "/" in str(s) else str(s)
        update_rows.append({
            "epoch": epoch,
            "question": f"{qid}...",
            "type": qtype,
            "action": upd["action"],
            "subject": short(triple[0]),
            "predicate": short(triple[1]),
            "object": short(triple[2]),
        })
    updates_df = pd.DataFrame(update_rows)
    print(f"Total belief updates: {len(updates_df)}")
    print(f"  Adds:     {(updates_df['action'] == 'add').sum()}")
    print(f"  Retracts: {(updates_df['action'] == 'retract').sum()}\n")
    display(updates_df.style.hide(axis="index"))
else:
    print("No belief updates recorded (imitation may be disabled).")

## Sample Interactions: First and Last Epoch

Full side-by-side view of the first 5 probes from epoch 0 (before learning) and the same questions from the final epoch (after learning), to see how B's responses evolved.

In [ ]:
epoch_0_probes = sorted([p for p in trajectory.probes if p.epoch == 0],
                        key=lambda p: p.probe_index)
last_epoch = max(p.epoch for p in trajectory.probes)
last_epoch_probes = {p.question_id: p for p in trajectory.probes if p.epoch == last_epoch}

# Show first 5 from epoch 0 paired with same question from last epoch
html = "<h4>Epoch 0 (before learning) vs Epoch {} (after learning)</h4>".format(last_epoch)

for p0 in epoch_0_probes[:5]:
    p_last = last_epoch_probes.get(p0.question_id)
    html += f'<div style="margin:16px 0; border-bottom: 1px solid #222; padding-bottom:12px">'
    html += f'<div style="color:#aaa; font-size:0.9em; margin-bottom:4px"><strong>Epoch 0:</strong></div>'
    html += render_probe_card(p0, question_map)
    if p_last:
        html += f'<div style="color:#aaa; font-size:0.9em; margin-bottom:4px; margin-top:8px"><strong>Epoch {last_epoch}:</strong></div>'
        html += render_probe_card(p_last, question_map)
    html += "</div>"

display(HTML(html))

## Per-Question Divergence Summary

In [ ]:
by_question = trajectory.divergence_by_question()

fig, ax = plt.subplots(figsize=(14, 5))
bq = by_question.reset_index()
bq_sorted = bq.sort_values("divergence_rate", ascending=False)
colors = [TYPE_COLORS.get(t, "#888") for t in bq_sorted["question_type"]]

ax.bar(range(len(bq_sorted)), bq_sorted["divergence_rate"], color=colors)
ax.set_xticks(range(len(bq_sorted)))
ax.set_xticklabels(bq_sorted["question_id"].str[:8] + "...", rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Divergence Rate (across epochs)")
ax.set_ylim(0, 1.05)
ax.set_title("Per-Question A-B Divergence Rate")
ax.legend(handles=[Patch(color=c, label=TYPE_LABELS[k]) for k, c in TYPE_COLORS.items()])
plt.tight_layout()
plt.show()

display(by_question)

## Interpretation

**What the data actually shows:**

This experiment is a feasibility proof. The machinery works — we can probe two operators with ontology-grounded questions, measure divergence, and track belief updates — but the results reveal several issues that need to be addressed before drawing conclusions about convergence or imitation learning.

**A is not stationary.** The design assumes A's behavior resets each epoch (history cleared), making it an approximately stationary operator so that changes in divergence can be attributed to B's learning. In practice, A (llama3.2:1b) degrades severely by epoch 4: parse failures jump from 1/25 in epoch 0 to 14/25 in epoch 4, and A's consistency among parseable responses drops from 75% to 45%. This is likely context overflow within the Ollama session — the conversation history is not being fully cleared, or the model's context window is saturating from the cumulative prompt length across 25 questions per epoch. The apparent "convergence" in the divergence curve (epoch 4 divergence drops) is an artifact: A is producing unparseable refusals ("I can't provide information on this topic"), which don't register as divergences.

**B's imitation is a wash.** Among the 21 questions that received at least one belief update, B improved on 7, degraded on 8, and stayed the same on 6 (epoch 0 vs epoch 4). The imitation mechanism is learning A's errors as readily as its correct assertions. Critically, B retracts correct beliefs when A gets the answer wrong — 15 retracts occurred on probes where B was already correct. The current update rule (trust A unconditionally on divergence) has no epistemic filter.

**Scoring agreement masks qualitative divergence.** When both operators score "correct" on the same probe, their responses are qualitatively incommensurable. A produces multi-paragraph natural language explanations (often with hallucinated detail), while B produces terse graph-lookup sentences like "Yes, vaccination is a vaccination." The binary score captures whether the yes/no polarity matches ground truth, but says nothing about the semantic content of the response. Two operators can "agree" on score while operating in entirely different representational regimes.

**What works:** The probe infrastructure, question bank, and per-type decomposition are sound. The heatmap and per-question divergence views are genuinely informative for identifying which ontological regions are hard for A. The belief update log provides a transparent record of B's state changes. As a feasibility proof, this demonstrates that the measurement apparatus is viable — the issues are in the experimental controls and the imitation policy, not the scaffolding.

## Proposed Next Steps

These are ordered by what would most improve experimental validity, not by difficulty.

### 1. Fix A's stationarity (blocking)
The context overflow problem invalidates epoch-over-epoch comparisons. Options:
- **Verify history reset**: Confirm that `OllamaClient` is actually clearing conversation history between epochs, not just between runs. If the Ollama `/api/chat` endpoint is accumulating messages across calls within an epoch, that's 25 long responses worth of context by epoch's end.
- **Stateless per-probe calls**: Send each probe as an independent request with no conversation history. This sacrifices any within-epoch conditioning but guarantees stationarity. For the measurement problem (how does A behave on these questions?) stateless probing is arguably more appropriate anyway.
- **Larger context model**: llama3.2:1b has a 2k effective context. A model with 8k+ context (e.g. llama3.2:3b or mistral) would defer the overflow problem, though not eliminate it.

### 2. Add an epistemic filter to B's update rule
The current rule — "on divergence, B unconditionally adopts A's position" — is too naive. B retracts correct beliefs 15 times across 5 epochs. Options:
- **Confidence-gated updates**: Only update B when A's response parses cleanly AND scores consistent with ground truth. This turns imitation into "learn from A when A is right" rather than unconditional mimicry — a different (weaker) claim, but an honest one.
- **Majority-vote updates**: Accumulate A's responses across epochs before committing a belief update. Only add/retract after A consistently asserts the same thing N times.
- **Separate imitation from evaluation**: Run a pure-measurement pass (no updates) first to characterize both operators, then run imitation as a distinct phase. This avoids the observer effect where B's updates change the divergence surface mid-measurement.

### 3. Richer scoring beyond binary polarity
The yes/no score is a coarse instrument. A's verbose explanations and B's terse lookups can both be "correct" while communicating entirely different things. Near-term options:
- **Response-length and structure features**: Log A's response length, number of claims, and hedging language as metadata alongside the binary score. This doesn't change scoring but makes the qualitative gap visible in the data.
- **Claim extraction**: Parse A's response into individual factual claims and score each against the ontology. This is a harder problem but would make the comparison meaningful — "A made 8 claims, 5 are ontologically grounded, 2 are hallucinated, 1 is ambiguous."
- **Targeted prompting**: Constrain A's responses to be more structured (e.g., "Answer yes or no, then give one reason") so the comparison surface is more commensurable with B.

### 4. Experimental controls for the batch_25 baseline
The existing `batch_25` single-operator data (25 runs, 625 turns) shows position-dependent consistency decay and per-run parse failure spikes that are consistent with the same context overflow problem. A clean comparison experiment needs:
- Stateless A as baseline (addresses stationarity)
- A no-imitation B run as control (same sparse B, same probes, no updates)
- Multiple seeds to separate question-order effects from genuine learning

### 5. Scale the question bank
25 questions across 3 types is enough for feasibility but not for statistical power. The per-type sample sizes (12 CM, 8 PR, 5 DJ) are too small to draw reliable per-type conclusions. The ontology has considerably more structure to draw from — expanding to 50-100 questions would improve coverage and enable within-type stratification.